# ML Pipeline Orchestration: Prefect, Airflow & ZenML

## Why Orchestration?
ML pipelines have multiple dependent steps: data ingestion → preprocessing → training → evaluation → deployment. Orchestration tools manage scheduling, dependencies, retries, monitoring, and parallelism.

## Tool Comparison
| Tool | Philosophy | Best For |
|------|-----------|----------|
| **Apache Airflow** | DAG-first, mature, Java-era feel | Complex enterprise pipelines |
| **Prefect** | Python-first, modern, flexible | Data engineers & ML teams |
| **ZenML** | ML-first, stack-based | MLOps pipelines with artifact tracking |
| **Kubeflow Pipelines** | Kubernetes-native | Large-scale K8s environments |
| **Metaflow** | Data science-first (Netflix) | Data scientists who want simplicity |

## Prefect

Prefect is a modern Python-native orchestration tool. Write regular Python, add decorators.

### Core Concepts
- **Flow**: Python function decorated with `@flow` the pipeline
- **Task**: Python function decorated with `@task` a unit of work
- **Deployment**: A flow packaged for scheduling/remote execution
- **Work Pool**: Infrastructure definition (process, Docker, K8s)
- **Block**: Reusable configuration (credentials, storage)

In [1]:
# pip install prefect
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import json, os

print('Prefect imported')

Prefect imported


In [2]:
# Define tasks
@task(name="Load Data", retries=3, retry_delay_seconds=10)
def load_data():
    """Load breast cancer dataset."""
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    print(f'Loaded {len(df)} samples, {len(data.feature_names)} features')
    return df

@task(name="Preprocess Data", cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def preprocess_data(df: pd.DataFrame, test_size: float = 0.2):
    """Split and scale data."""
    feature_cols = [c for c in df.columns if c != 'target']
    X = df[feature_cols].values
    y = df['target'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    print(f'Train: {len(X_train)}, Test: {len(X_test)}')
    return X_train, X_test, y_train, y_test

@task(name="Train Model")
def train_model(X_train, y_train, n_estimators: int = 100, max_depth: int = 5):
    """Train Random Forest classifier."""
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    print(f'Model trained with {n_estimators} estimators')
    return model

@task(name="Evaluate Model")
def evaluate_model(model, X_test, y_test):
    """Evaluate model performance."""
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    metrics = {'accuracy': accuracy, 'f1_macro': report['macro avg']['f1-score']}
    print(f'Accuracy: {accuracy:.4f}')
    return metrics

@task(name="Save Metrics")
def save_metrics(metrics: dict, output_path: str = '/tmp/metrics.json'):
    """Save metrics to file."""
    with open(output_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f'Metrics saved to {output_path}')
    return output_path

# Define the flow
@flow(name="ML Training Pipeline", log_prints=True)
def ml_training_pipeline(test_size: float = 0.2, n_estimators: int = 100, max_depth: int = 5):
    """End-to-end ML training pipeline."""
    df = load_data()
    X_train, X_test, y_train, y_test = preprocess_data(df, test_size=test_size)
    model = train_model(X_train, y_train, n_estimators=n_estimators, max_depth=max_depth)
    metrics = evaluate_model(model, X_test, y_test)
    path = save_metrics(metrics)
    return metrics

# Run the flow
result = ml_training_pipeline(test_size=0.2, n_estimators=150, max_depth=7)
print(f'Pipeline result: {result}')

17:21:09.167 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8660
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

17:21:28.078 | INFO    | Flow run 'feathered-quail' - Beginning flow run 'feathered-quail' for flow 'ML Training Pipeline'

17:21:28.124 | ERROR   | GlobalEventLoopThread | prefect._internal.concurrency - Service 'EventsWorker' failed with 3 pending items.


17:21:28.137 | INFO    | Task run 'Load Data-304' - Loaded 569 samples, 30 features

17:21:28.142 | ERROR   | GlobalEventLoopThread | prefect._internal.concurrency - Service 'EventsWorker' failed with 0 pending items.


...

17:21:28.591 | INFO    | Task run 'Save Metrics-6ac' - Metrics saved to /tmp/metrics.json

17:21:28.596 | ERROR   | GlobalEventLoopThread | prefect._internal.concurrency - Service 'EventsWorker' failed with 0 pending items.


17:21:28.596 | INFO    | Task run 'Save Metrics-6ac' - Finished in state Completed()

17:21:29.122 | INFO    | Flow run 'feathered-quail' - Finished in state Completed()

Pipeline result: {'accuracy': 0.9473684210526315, 'f1_macro': 0.9434523809523809}


## Prefect Deployments & Scheduling

```python
from prefect.deployments import Deployment
from prefect.server.schemas.schedules import CronSchedule

# Create deployment
deployment = Deployment.build_from_flow(
    flow=ml_training_pipeline,
    name="daily-training",
    schedule=CronSchedule(cron="0 2 * * *"),  # 2 AM daily
    parameters={"n_estimators": 200},
    work_pool_name="default-agent-pool"
)
deployment.apply()
```

```bash
# Start Prefect server
prefect server start

# Start worker
prefect worker start --pool default-agent-pool

# Run flow from CLI
prefect deployment run 'ML Training Pipeline/daily-training'
```

## Apache Airflow

Airflow is the industry standard for workflow orchestration, using Python-defined DAGs.

### Core Concepts
- **DAG**: Directed Acyclic Graph the pipeline definition
- **Task**: A unit of work (Operator)
- **Operator**: Template for a task (PythonOperator, BashOperator, etc.)
- **XCom**: Cross-communication between tasks
- **Scheduler**: Triggers DAG runs per schedule
- **Executor**: How tasks run (LocalExecutor, CeleryExecutor, KubernetesExecutor)

In [3]:
# Airflow DAG example (save to ~/airflow/dags/ml_pipeline_dag.py)
AIRFLOW_DAG = '''
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.utils.task_group import TaskGroup
import pandas as pd
import json

# Default arguments
default_args = {
    'owner': 'ml-team',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'email': ['alerts@company.com'],
    'email_on_failure': True,
    'retries': 2,
    'retry_delay': timedelta(minutes=5),
}

# Define DAG
with DAG(
    dag_id='ml_training_pipeline',
    default_args=default_args,
    description='Daily ML model retraining',
    schedule_interval='@daily',
    catchup=False,
    tags=['ml', 'training'],
) as dag:

    # Task functions
    def _extract_data(**kwargs):
        from sklearn.datasets import load_breast_cancer
        data = load_breast_cancer()
        df = pd.DataFrame(data.data, columns=data.feature_names)
        df['target'] = data.target
        df.to_csv('/tmp/raw_data.csv', index=False)
        kwargs['ti'].xcom_push(key='data_path', value='/tmp/raw_data.csv')  # XCom
        return '/tmp/raw_data.csv'

    def _preprocess(**kwargs):
        ti = kwargs['ti']
        data_path = ti.xcom_pull(key='data_path', task_ids='extract_data')
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import StandardScaler
        df = pd.read_csv(data_path)
        # ... preprocessing ...
        return 'preprocessed'

    def _train_model(**kwargs):
        from sklearn.ensemble import RandomForestClassifier
        import joblib
        # ... training ...
        print("Model trained")

    def _evaluate(**kwargs):
        # ... evaluation ...
        print("Model evaluated")

    def _notify_success(**kwargs):
        print("Pipeline completed successfully! Sending notification...")

    # Define tasks
    with TaskGroup("data_preparation") as data_prep:
        extract = PythonOperator(task_id="extract_data", python_callable=_extract_data)
        preprocess = PythonOperator(task_id="preprocess_data", python_callable=_preprocess)
        extract >> preprocess

    train = PythonOperator(task_id="train_model", python_callable=_train_model)
    evaluate = PythonOperator(task_id="evaluate_model", python_callable=_evaluate)
    notify = PythonOperator(task_id="notify", python_callable=_notify_success)

    # Define dependencies
    data_prep >> train >> evaluate >> notify
'''

print("Airflow DAG code:")
print(AIRFLOW_DAG[:500] + '...')

Airflow DAG code:

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.utils.task_group import TaskGroup
import pandas as pd
import json

# Default arguments
default_args = {
    'owner': 'ml-team',
    'depends_on_past': False,
    'start_date': datetime(2024, 1, 1),
    'email': ['alerts@company.com'],
    'email_on_failure': True,
    'retries': 2,
    'retry_delay': timedelta(minutes=5...


## Airflow Setup

```bash
# Install
pip install apache-airflow apache-airflow-providers-amazon

# Initialize DB
airflow db init

# Create admin user
airflow users create --username admin --firstname Admin --lastname Admin \
  --role Admin --email admin@example.com --password admin

# Start webserver and scheduler
airflow webserver --port 8080 &
airflow scheduler &

# Trigger DAG manually
airflow dags trigger ml_training_pipeline

# Check task logs
airflow tasks logs ml_training_pipeline train_model 2024-01-01
```

## ZenML

ZenML is an MLOps framework that defines pipelines as Python code and supports multiple backends (stacks).

```python
from zenml import pipeline, step
from zenml.config import DockerSettings

@step
def load_data() -> pd.DataFrame:
    data = load_breast_cancer()
    return pd.DataFrame(data.data, columns=data.feature_names)

@step
def train(data: pd.DataFrame) -> RandomForestClassifier:
    model = RandomForestClassifier(n_estimators=100)
    model.fit(data.drop('target', axis=1), data['target'])
    return model

@pipeline
def ml_pipeline():
    data = load_data()
    model = train(data)

ml_pipeline()
```

### ZenML Stack Components
```bash
zenml stack describe   # current stack
zenml stack list       # all stacks
zenml artifact-store register s3_store --flavor=s3 --path=s3://bucket
zenml model-deployer register seldon --flavor=seldon
```

## Additional Learning Resources

### Prefect
- [Prefect Docs](https://docs.prefect.io/) Complete reference
- [Prefect Quickstart](https://docs.prefect.io/latest/getting-started/quickstart/)
- [Prefect YouTube](https://www.youtube.com/@PrefectIO)

### Airflow
- [Airflow Docs](https://airflow.apache.org/docs/) Complete reference
- [Astronomer Airflow Guides](https://docs.astronomer.io/learn) Best Airflow tutorials
- [Airflow: The Hands-On Guide](https://www.udemy.com/course/the-complete-hands-on-course-to-master-apache-airflow/)

### ZenML
- [ZenML Docs](https://docs.zenml.io/) Complete reference
- [ZenML Blog](https://www.zenml.io/blog)

### Papers
- [Sculley et al. Hidden Technical Debt in ML Systems](https://papers.nips.cc/paper/2015/file/86df7dcfd896fcaf2674f757a2463eba-Paper.pdf)